# HealthConnect Week 7 — Actual Data Analytics Validation
This notebook uses the 19 uploaded source files. The analysis preserves original tables and reports join failures rather than silently repairing them.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
base=Path('.')
appointments=pd.read_excel(base/'HC_Clinical_Transactions.xlsx',sheet_name='Appointments')
patients=pd.read_csv(base/'HC_Patients.csv')
billing=pd.read_csv(base/'HC_Billing.csv')
payments=pd.read_csv(base/'HC_Payments.csv')
feedback=pd.read_csv(base/'HC_Patient_Feedback.csv')
diagnoses=pd.read_csv(base/'HC_Diagnoses.csv')
departments=pd.read_csv(base/'HC_Departments.csv')


## Data inventory and quality checks

In [ ]:
inventory=[]
for f in sorted(base.glob('*.csv')):
    d=pd.read_csv(f); inventory.append({'file':f.name,'rows':len(d),'columns':len(d.columns),'missing_cells':int(d.isna().sum().sum()),'duplicates':int(d.duplicated().sum())})
pd.DataFrame(inventory)


## Appointment relationship checks

In [ ]:
checks={
 'billing_unmatched':int((~billing.AppointmentID.isin(appointments.AppointmentID)).sum()),
 'diagnosis_csv_unmatched':int((~diagnoses.AppointmentID.isin(appointments.AppointmentID)).sum()),
 'feedback_unmatched':int((~feedback.AppointmentID.isin(appointments.AppointmentID)).sum()),
 'facility_department_conflicts':int((appointments.FacilityID!=appointments.DepartmentID.map(departments.set_index('DepartmentID').FacilityID)).sum())}
checks


## Table-bounded descriptive analysis

In [ ]:
patients.Gender.value_counts(), appointments.VisitType.value_counts(), feedback.Rating.value_counts().sort_index(), feedback.Rating.mean()


## Finance reconciliation

In [ ]:
bp=billing.merge(payments,on='BillingID',how='left')
{'billing_total':billing.Amount.sum(),'payments_total':payments.AmountPaid.sum(),'amount_mismatches':int((bp.Amount!=bp.AmountPaid).sum())}


## Interpretation safeguards
Do not join billing, diagnoses or feedback to the appointment sheet until the authoritative appointment population is confirmed. Do not infer bed occupancy from a file where every bed is Available. Do not infer sentiment from feedback comments while comment labels conflict with numeric ratings.